# Impute solvents from a processed NPZ

This notebook loads a processed structure `.npz`, optionally strips existing solvents, imputes additional solvents with `Structure.impute_solvents(...)`, and writes the result as an mmCIF file.

Edit the config cell below, then run the notebook top to bottom.

In [105]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import numpy as np
from boltzgen.data.data import Structure
from boltzgen.data.write.mmcif import to_mmcif


BOLTZGEN_SRC = Path("/data/rbg/users/gloriama/dev/foldeverything/src")
NOTEBOOK_ROOT = Path("/data/rbg/users/gloriama/dev/water_conservation")


from gloria_hbond_helpers import gloria_get_solvent_hbond_counts_and_mask, gloria_get_solvent_hbond_counts, gloria_get_solvent_hbond_mask, gloria_remove_weak_solvents
# from unused_solvent_filters import 
from basic_helpers import resolve_npz_path, raw_gt_structure, stripped_gt_structure, filtered_gt_structure, count_stripped_gt_residues, count_residues
from impute_solvents_from_triples import impute_solvents_from_atom_triples, filter_solvent_clashes
from recall import recall_result





The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [71]:
import importlib
import gloria_hbond_helpers
import impute_solvents_from_triples
import recall
import basic_helpers

importlib.reload(gloria_hbond_helpers)
importlib.reload(impute_solvents_from_triples)
importlib.reload(recall)
importlib.reload(basic_helpers)


<module 'basic_helpers' from '/data/rbg/users/gloriama/dev/water_conservation/imputation/basic_helpers.py'>

In [72]:
# Config 
NPZ_ROOT = Path("/data/rbg/shared/datasets/processed_rcsb/rcsb_solvents/structures")
NPZ_PATH = None  # Set to an explicit file path to override PDB_ID + NPZ_ROOT.
OUTPUT_DIR = NOTEBOOK_ROOT / "outputs"
ONE_SOLVENT_PER_CHAIN = True


# Unused for new imputation functions
MIN_SOLVENTS = 20
STRIP_SOLVENTS_BEFOREHAND = True
ALLOW_OVERLAP_WITH_REAL_SOLVENTS = False
MAX_SAMPLE_ATTEMPTS = 1000


# Change 
PDB_ID = "1ubq"
COLLISION_MIN_DIST = 2.5 # ? 
TRIPLE_HBOND_PAIR_MAX_DIST = 5.6
# TRIPLE_HBOND_PAIR_MAX_DIST = 8.0
npz_path = resolve_npz_path(PDB_ID, NPZ_ROOT, NPZ_PATH)
gt_structure = Structure.load(npz_path)
gt_structure = gt_structure.to_one_solvent_per_chain(gt_structure)
gt_solvent_count = gt_structure.count_solvents()
stripped_gt = gt_structure.remove_solvents()
gt_solvent_count


58

# Save cifs (gt without imputation, with filtering.)

In [ ]:
default_pdb_id = "1ubq"
default_npz_path = resolve_npz_path(default_pdb_id, NPZ_ROOT)
default_output_dir = Path(OUTPUT_DIR)
default_output_dir.mkdir(parents=True, exist_ok=True)

default_structure = Structure.load(default_npz_path)
default_output_path = default_output_dir / f"{default_pdb_id}_default.cif"
default_output_path.write_text(to_mmcif(default_structure))

print(f"Input NPZ: {default_npz_path}")
print(f"Wrote CIF to: {default_output_path}")

In [ ]:
weak_pdb_id = "1ubq"
weak_npz_path = resolve_npz_path(weak_pdb_id, NPZ_ROOT)
weak_output_dir = Path(OUTPUT_DIR)
weak_output_dir.mkdir(parents=True, exist_ok=True)
weak_structure = Structure.load(weak_npz_path)
weak_structure = weak_structure.to_one_solvent_per_chain(weak_structure)
weak_filtered_structure = gloria_remove_weak_solvents(weak_structure)
weak_output_path = weak_output_dir / f"{weak_pdb_id}_weak_filtered.cif"
weak_output_path.write_text(to_mmcif(weak_filtered_structure))
print(f"Input NPZ: {weak_npz_path}")
print(f"Original solvent count: {weak_structure.count_solvents()}")
print(f"Filtered solvent count: {weak_filtered_structure.count_solvents()}")
print(f"Wrote CIF to: {weak_output_path}")


In [ ]:
weak3_pdb_id = "1ubq"
weak3_npz_path = resolve_npz_path(weak3_pdb_id, NPZ_ROOT)
weak3_output_dir = Path(OUTPUT_DIR)
weak3_output_dir.mkdir(parents=True, exist_ok=True)
weak3_structure = Structure.load(weak3_npz_path)
weak3_structure = weak3_structure.to_one_solvent_per_chain(weak3_structure)
weak3_filtered_structure = gloria_remove_weak_solvents(
    weak3_structure,
    min_hbonds=3,
)
weak3_output_path = weak3_output_dir / f"{weak3_pdb_id}_weak_filtered_min3.cif"
weak3_output_path.write_text(to_mmcif(weak3_filtered_structure))
print(f"Input NPZ: {weak3_npz_path}")
print(f"Original solvent count: {weak3_structure.count_solvents()}")
print(f"Filtered solvent count (min_hbonds=3): {weak3_filtered_structure.count_solvents()}")
print(f"Wrote CIF to: {weak3_output_path}")


In [35]:
weak3_pdb_id = "7zzh"
weak3_npz_path = resolve_npz_path(weak3_pdb_id, NPZ_ROOT)
weak3_output_dir = Path(OUTPUT_DIR)
weak3_output_dir.mkdir(parents=True, exist_ok=True)
weak3_structure = Structure.load(weak3_npz_path)
weak3_structure = weak3_structure.to_one_solvent_per_chain(weak3_structure)
weak3_filtered_structure = gloria_remove_weak_solvents(
    weak3_structure,
    min_hbonds=3,
)
gt_7zzh_3hbonds = weak3_filtered_structure
weak3_output_path = weak3_output_dir / f"{weak3_pdb_id}_weak_filtered_min3.cif"
weak3_output_path.write_text(to_mmcif(weak3_filtered_structure))
print(f"Input NPZ: {weak3_npz_path}")
print(f"Original solvent count: {weak3_structure.count_solvents()}")
print(f"Filtered solvent count (min_hbonds=3): {weak3_filtered_structure.count_solvents()}")
print(f"Wrote CIF to: {weak3_output_path}")


Input NPZ: /data/rbg/shared/datasets/processed_rcsb/rcsb_solvents/structures/7zzh.npz
Original solvent count: 236
Filtered solvent count (min_hbonds=3): 72
Wrote CIF to: /data/rbg/users/gloriama/dev/water_conservation/outputs/7zzh_weak_filtered_min3.cif


## Old imputation function (sample 3, 2, 1, etc.)
- Bug: was only attempting starting at 2, 1, etc. 
- For some reason only scores 1hbond waters, even with max_sample_attempts = 1000. 

In [4]:
npz_path = resolve_npz_path(PDB_ID, NPZ_ROOT, NPZ_PATH)
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

if not npz_path.exists():
    raise FileNotFoundError(f"Could not find NPZ input: {npz_path}")

structure = Structure.load(npz_path)
loaded_solvents = structure.count_solvents()

if STRIP_SOLVENTS_BEFOREHAND:
    structure = structure.remove_solvents()

post_strip_solvents = structure.count_solvents()
structure = impute_solvents_with_num_hbonds(
    structure,
    one_solvent_per_chain=ONE_SOLVENT_PER_CHAIN,
    min_solvents=MIN_SOLVENTS,
    interaction_min_dist=INTERACTION_MIN_DIST,
    max_sample_attempts=MAX_SAMPLE_ATTEMPTS,
    allow_overlap_with_real_solvents=ALLOW_OVERLAP_WITH_REAL_SOLVENTS,
)
final_solvents = structure.count_solvents()

output_path = output_dir / f"{PDB_ID.lower()}_imputed_first_stripped.cif"
output_path.write_text(to_mmcif(structure))

print(f"Input NPZ: {npz_path}")
print(f"Loaded solvent count: {loaded_solvents}")
print(f"Post-strip solvent count: {post_strip_solvents}")
print(f"Final solvent count: {final_solvents}")
print(f"Wrote CIF to: {output_path}")


sample_attempts=10003 until a n_function(sample_attempts)=1 h-bond thing was found, for the impute_idx=0th water.
sample_attempts=10001 until a n_function(sample_attempts)=1 h-bond thing was found, for the impute_idx=1th water.


KeyboardInterrupt: 

In [ ]:
# Example helper calls
example_structure = Structure.load(npz_path)
example_structure = example_structure.to_one_solvent_per_chain(example_structure)

gloria_hbond_counts = gloria_get_solvent_hbond_counts(example_structure)
(
    gloria_hbond_counts_full,
    gloria_present_solvent_chain_mask,
) = gloria_get_solvent_hbond_counts_and_mask(example_structure)
gloria_keep_solvent_mask = gloria_get_solvent_hbond_mask(
    example_structure,
    min_hbonds=2,
)

gloria_weak_filtered_structure = gloria_remove_weak_solvents(
    example_structure,
    min_hbonds=2,
)
gloria_bfactor_filtered_structure = gloria_remove_low_b_factor_solvents(
    example_structure,
    n_keep=10,
)

print(f"Present solvent chains: {gloria_present_solvent_chain_mask.sum()}")
print(f"Per-chain hbond counts shape: {gloria_hbond_counts.shape}")
print(
    "Counts agree across helper variants:",
    (gloria_hbond_counts == gloria_hbond_counts_full).all(),
)
print(f"Chains meeting min_hbonds=2: {gloria_keep_solvent_mask.sum()}")
print(
    f"Solvents after gloria_remove_weak_solvents: {gloria_weak_filtered_structure.count_solvents()}"
)
print(
    f"Solvents after gloria_remove_low_b_factor_solvents(n_keep=10): {gloria_bfactor_filtered_structure.count_solvents()}"
)

# Test recall

In [ ]:

# If the notebook is running from the repo root:

# If it's running from inside `imputation/`, use this instead:
# from recall import compare_structure_file_waters


result = compare_structure_waters(
    gt_7zzh_3hbonds,
    final_7zzh_imputed,
    cutoff=1.0,  # angstroms
)

print("waters in A:", result.num_waters_a)
print("waters in B:", result.num_waters_b)
print("matching waters in A:", result.num_matching_waters)
print("recall:", result.recall)

# Boolean mask over waters in A
print("match mask:", result.match_mask)

# For matched waters in A, index of nearest water in B
print("nearest indices in B:", result.nearest_indices_in_b)

# Distances for matched waters; unmatched entries are inf
print("nearest distances:", result.nearest_distances)

waters in A: 72
waters in B: 890
matching waters in A: 36
recall: 0.5
match mask: [False False False False  True  True  True  True False False False False
  True  True  True  True  True  True  True  True False False False False
 False False False False  True  True  True  True False False False False
  True  True  True  True  True  True  True  True False False False False
  True  True  True  True False False False False False False False False
 False False False False  True  True  True  True  True  True  True  True]
nearest indices in B: [ -1  -1  -1  -1 429 192 400 158  -1  -1  -1  -1 780 573 338  96 721 514
 277  35  -1  -1  -1  -1  -1  -1  -1  -1 777 570 335  93  -1  -1  -1  -1
 803 596 361 119 778 571 336  94  -1  -1  -1  -1 723 516 279  37  -1  -1
  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1 851 647 431 194 746 539 302  60]
nearest distances: [       inf        inf        inf        inf 0.15155723 0.15155723
 0.15155718 0.15155543        inf        inf        inf        inf
 0.46763536

# Impute and save pipeline

In [ ]:
PDB_ID = "1ubq"
TRIPLE_HBOND_PAIR_MAX_DIST = "NOOOOOOOO"
MAX_HBOND_LENGTH = 7.0
PROTEIN_CLASH_RADIUS = 2.5
SOLVENT_CLASH_RADIUS = 0.0

output_path = OUTPUT_DIR / (
    f"token_hbonds_{PDB_ID}_clash_{PROTEIN_CLASH_RADIUS}_{SOLVENT_CLASH_RADIUS}_max_hbond_length_{MAX_HBOND_LENGTH}.cif"
)

# load in stripped gt structure 
start = perf_counter()
gt_structure = stripped_gt_structure(PDB_ID)
strip_time = perf_counter() - start

# impute waters from atom triples
start = perf_counter()
imputed = impute_solvents_from_atom_triples(
    gt_structure,
    max_hbond_length=MAX_HBOND_LENGTH,
)
impute_time = perf_counter() - start

# filter clashes 
start = perf_counter()
no_collisions = filter_solvent_clashes(
    imputed,
    protein_clash_rad=PROTEIN_CLASH_RADIUS,
    solvent_clash_rad=SOLVENT_CLASH_RADIUS,
)
clash_time = perf_counter() - start

# optionally, remove weak waters. with min_hbonds=3, this filters out not many waters. 

# start = perf_counter()
# weak_removed_from_imputed = gloria_remove_weak_solvents(no_collisions, min_hbonds=3)
# final_solvent_count = weak_removed_from_imputed.count_solvents()
# final_filter_time = perf_counter() - start



start = perf_counter()
output_path.write_text(to_mmcif(no_collisions))
write_time = perf_counter() - start


print(f"Input NPZ: {npz_path}")
print(f"Placed waters: {imputed.count_solvents()}")
print(f"Placed waters after clashes: {no_collisions.count_solvents()}")
# print(f"After weak solvent filtering: {final_solvent_count}")


print(f"Time to strip structure: {strip_time:.2f}s")
print(f"Time to impute waters: {impute_time:.2f}s")
print(f"Time to filter clashes: {clash_time:.2f}s")
# print(f"Time to filter weak waters: {final_filter_time:.2f}s")
print(f"Time to write CIF: {write_time:.2f}s")
print(f"Wrote CIF to: {output_path}")


Raw GT structure number of solvents: 58
Number of candidate atoms: 223
Number of neighbor lists: 223
Neighbor lists: [[0, 1, 2, 3, 43, 44, 45, 47, 48], [0, 1, 2, 3, 4, 7, 38, 39, 40, 43, 44, 45], [0, 1, 2, 3, 4, 44, 49, 177, 178, 179, 182, 185], [0, 1, 2, 3, 4, 6, 7, 38, 43, 44, 185, 188], [1, 2, 3, 4, 6, 7, 8, 9, 38, 179, 182, 183, 185, 186, 188, 189, 190], [5, 6, 8, 36], [3, 4, 5, 6, 7, 188], [1, 3, 4, 6, 7, 8, 9, 37, 38, 39, 43, 185, 189, 190], [4, 5, 7, 8, 9, 10, 11, 33, 34, 35, 36, 37, 38, 39, 190], [4, 7, 8, 9, 10, 11, 33, 37, 38, 186, 189, 190, 192, 194, 195]]
neighbor_list_time=0.00s
place_water_time=0.07s
Number of triples: 2103
Number of placed waters: 1902
append_imputed_solvents_time=0.00s
Number of surviving waters after clash with protein: 71
Number of surviving waters after clash with other waters: 71
Input NPZ: /data/rbg/shared/datasets/processed_rcsb/rcsb_solvents/structures/1ubq.npz
Triple max pair distance: 5.6
Placed waters: 1902
Placed waters after clashes: 71
Time

In [67]:
PDB_ID = "1ubq"
TRIPLE_HBOND_PAIR_MAX_DIST = "NOOOOOOOO"
MAX_HBOND_LENGTH = 2.8
PROTEIN_CLASH_RADIUS = 2.5
SOLVENT_CLASH_RADIUS = 0.0

output_path = OUTPUT_DIR / (
    f"token_hbonds_{PDB_ID}_clash_{PROTEIN_CLASH_RADIUS}_{SOLVENT_CLASH_RADIUS}_max_hbond_length_{MAX_HBOND_LENGTH}.cif"
)

global_start = perf_counter()
# load in stripped gt structure 
start = perf_counter()
gt_structure = stripped_gt_structure(PDB_ID)
strip_time = perf_counter() - start

# impute waters from atom triples
start = perf_counter()
imputed = impute_solvents_from_atom_triples(
    gt_structure,
    max_hbond_length=MAX_HBOND_LENGTH,
)
impute_time = perf_counter() - start

# filter clashes 
start = perf_counter()
no_collisions = filter_solvent_clashes(
    imputed,
    protein_clash_rad=PROTEIN_CLASH_RADIUS,
    solvent_clash_rad=SOLVENT_CLASH_RADIUS,
)
clash_time = perf_counter() - start

# optionally, remove weak waters. with min_hbonds=3, this filters out not many waters. 

# start = perf_counter()
# weak_removed_from_imputed = gloria_remove_weak_solvents(no_collisions, min_hbonds=3)
# final_solvent_count = weak_removed_from_imputed.count_solvents()
# final_filter_time = perf_counter() - start



start = perf_counter()
output_path.write_text(to_mmcif(no_collisions))
write_time = perf_counter() - start

global_time = perf_counter() - global_start

print(f"Input NPZ: {npz_path}")
print(f"Placed waters: {imputed.count_solvents()}")
print(f"Placed waters after clashes: {no_collisions.count_solvents()}")
# print(f"After weak solvent filtering: {final_solvent_count}")


print(f"Time to strip structure: {strip_time:.2f}s")
print(f"Time to impute waters: {impute_time:.2f}s")
print(f"Time to filter clashes: {clash_time:.2f}s")
# print(f"Time to filter weak waters: {final_filter_time:.2f}s")
print(f"Time to write CIF: {write_time:.2f}s")
print(f"Wrote CIF to: {output_path}")
print(f"Total time: {global_time:.2f}s")


Raw GT structure number of solvents: 58
Number of candidate atoms: 223
Number of neighbor lists: 223
Neighbor lists: [[0, 1, 2, 3, 43, 44, 45, 47, 48], [0, 1, 2, 3, 4, 7, 38, 39, 40, 43, 44, 45], [0, 1, 2, 3, 4, 44, 49, 177, 178, 179, 182, 185], [0, 1, 2, 3, 4, 6, 7, 38, 43, 44, 185, 188], [1, 2, 3, 4, 6, 7, 8, 9, 38, 179, 182, 183, 185, 186, 188, 189, 190], [5, 6, 8, 36], [3, 4, 5, 6, 7, 188], [1, 3, 4, 6, 7, 8, 9, 37, 38, 39, 43, 185, 189, 190], [4, 5, 7, 8, 9, 10, 11, 33, 34, 35, 36, 37, 38, 39, 190], [4, 7, 8, 9, 10, 11, 33, 37, 38, 186, 189, 190, 192, 194, 195]]
neighbor_list_time=0.00s
place_water_time=0.08s
Number of triples: 2103
Number of placed waters: 1902
append_imputed_solvents_time=0.00s
Number of surviving waters after clash with protein: 71
Number of surviving waters after clash with other waters: 71
Input NPZ: /data/rbg/shared/datasets/processed_rcsb/rcsb_solvents/structures/1ubq.npz
Placed waters: 1902
Placed waters after clashes: 71
Time to strip structure: 0.01s
Tim

In [ ]:
PDB_ID = "8hig"
TRIPLE_HBOND_PAIR_MAX_DIST = "NOOOOOOOO"
MAX_HBOND_LENGTH = 3.5
PROTEIN_CLASH_RADIUS = 2.2
SOLVENT_CLASH_RADIUS = 3.0

output_path = OUTPUT_DIR / (
    f"token_hbonds_{PDB_ID}_clash_{PROTEIN_CLASH_RADIUS}_{SOLVENT_CLASH_RADIUS}_max_hbond_length_{MAX_HBOND_LENGTH}.cif"
)

global_start = perf_counter()
# load in stripped gt structure 
start = perf_counter()
gt_structure = stripped_gt_structure(PDB_ID)
strip_time = perf_counter() - start

# impute waters from atom triples
start = perf_counter()
imputed = impute_solvents_from_atom_triples(
    gt_structure,
    max_hbond_length=MAX_HBOND_LENGTH,
)
impute_time = perf_counter() - start

# filter clashes 
start = perf_counter()
no_collisions = filter_solvent_clashes(
    imputed,
    protein_clash_rad=PROTEIN_CLASH_RADIUS,
    solvent_clash_rad=SOLVENT_CLASH_RADIUS,
)
clash_time = perf_counter() - start

# optionally, remove weak waters. with min_hbonds=3, this filters out not many waters. 

# start = perf_counter()
# weak_removed_from_imputed = gloria_remove_weak_solvents(no_collisions, min_hbonds=3)
# final_solvent_count = weak_removed_from_imputed.count_solvents()
# final_filter_time = perf_counter() - start



start = perf_counter()
output_path.write_text(to_mmcif(no_collisions))
write_time = perf_counter() - start

global_time = perf_counter() - global_start

print(f"Placed waters: {imputed.count_solvents()}")
print(f"Placed waters after clashes: {no_collisions.count_solvents()}")
# print(f"After weak solvent filtering: {final_solvent_count}")


print(f"Time to strip structure: {strip_time:.2f}s")
print(f"Time to impute waters: {impute_time:.2f}s")
print(f"Time to filter clashes: {clash_time:.2f}s")
# print(f"Time to filter weak waters: {final_filter_time:.2f}s")
print(f"Time to write CIF: {write_time:.2f}s")
print(f"Wrote CIF to: {output_path}")
print(f"Total time: {global_time:.2f}s")

Raw GT structure number of solvents: 48
Number of candidate atoms: 921
Number of neighbor lists: 921
Neighbor lists: [[0, 1, 2, 3, 4, 6, 8, 30, 38], [0, 1, 2, 3, 4, 6, 38], [0, 1, 2, 3, 4, 5, 6, 7, 8, 30, 36, 38], [0, 1, 2, 3, 4, 5, 6, 7, 34, 38], [0, 1, 2, 3, 4, 5, 6, 7, 9, 27, 30, 32, 34, 35, 36, 38], [2, 3, 4, 5, 6, 7, 8, 9, 10, 26, 27, 28, 29, 30, 32, 33, 34, 35, 36, 38, 110], [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 27, 28, 29, 30, 32, 34, 36, 38], [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 26, 27, 28, 30, 32], [0, 2, 5, 6, 7, 8, 9, 10, 11, 13, 26, 27, 28, 30, 31, 54], [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 23, 25, 26, 27, 28, 30, 32, 34]]
neighbor_list_time=0.00s
place_water_time=1.18s
Number of triples: 41251
Number of placed waters: 19364
append_imputed_solvents_time=0.04s
Number of surviving waters after clash with protein: 1239
Number of surviving waters after clash with other waters: 240
Input NPZ: /data/rbg/shared/datasets/processed_rcsb/rcsb_solvents/structures/1ubq.npz
Placed waters:

In [109]:
PDB_ID = "8hig"
gt_output_path = OUTPUT_DIR / (
    f"gt_{PDB_ID}_3hbonds.cif"
)

gt_structure = filtered_gt_structure(PDB_ID, min_hbonds=3)
gt_output_path.write_text(to_mmcif(gt_structure))
print(f"Wrote GT structure to: {gt_output_path}")

Raw GT structure number of solvents: 48
Filtered (>=3 H-bonds) GT structure number of solvents: 5
Wrote GT structure to: /data/rbg/users/gloriama/dev/water_conservation/outputs/gt_8hig_3hbonds.cif


# Recall

In [77]:


example_result = recall_result("1ubq", recall_threshold=1.0, protein_clash_rad=2.5, solvent_clash_rad=0.0, max_hbond_length=2.8)
{
    "filtered_gt_solvents": example_result.num_waters_a,
    "imputed_solvents": example_result.num_waters_b,
    "matching_solvents": example_result.num_matching_waters,
    "recall": example_result.recall,
}


Number of candidate atoms: 223
Number of neighbor lists: 223
Neighbor lists: [[0, 1, 2, 3, 43, 44, 45, 47, 48], [0, 1, 2, 3, 4, 7, 38, 39, 40, 43, 44, 45], [0, 1, 2, 3, 4, 44, 49, 177, 178, 179, 182, 185], [0, 1, 2, 3, 4, 6, 7, 38, 43, 44, 185, 188], [1, 2, 3, 4, 6, 7, 8, 9, 38, 179, 182, 183, 185, 186, 188, 189, 190], [5, 6, 8, 36], [3, 4, 5, 6, 7, 188], [1, 3, 4, 6, 7, 8, 9, 37, 38, 39, 43, 185, 189, 190], [4, 5, 7, 8, 9, 10, 11, 33, 34, 35, 36, 37, 38, 39, 190], [4, 7, 8, 9, 10, 11, 33, 37, 38, 186, 189, 190, 192, 194, 195]]
neighbor_list_time=0.00s
place_water_time=0.08s
Number of triples: 2103
Number of placed waters: 1902
append_imputed_solvents_time=0.00s
Number of surviving waters after clash with protein: 71
Number of surviving waters after clash with other waters: 71


{'filtered_gt_solvents': 3,
 'imputed_solvents': 71,
 'matching_solvents': 1,
 'recall': 0.3333333333333333}

In [78]:
example_result = recall_result("1ubq", recall_threshold=0.5, protein_clash_rad=2.5, solvent_clash_rad=0.0, max_hbond_length=3.5)
{
    "filtered_gt_solvents": example_result.num_waters_a,
    "imputed_solvents": example_result.num_waters_b,
    "matching_solvents": example_result.num_matching_waters,
    "recall": example_result.recall,
}

Number of candidate atoms: 223
Number of neighbor lists: 223
Neighbor lists: [[0, 1, 2, 3, 4, 7, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49], [0, 1, 2, 3, 4, 7, 8, 37, 38, 39, 40, 41, 42, 43, 44, 45, 47, 48, 185], [0, 1, 2, 3, 4, 7, 43, 44, 45, 46, 47, 49, 177, 178, 179, 182, 183, 185, 189, 191], [0, 1, 2, 3, 4, 5, 6, 7, 8, 37, 38, 39, 40, 41, 43, 44, 179, 182, 183, 185, 188], [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 37, 38, 39, 43, 44, 179, 182, 183, 185, 186, 187, 188, 189, 190, 191], [3, 4, 5, 6, 7, 8, 34, 35, 36, 37, 38, 41, 188], [3, 4, 5, 6, 7, 8, 36, 38, 185, 187, 188], [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 33, 34, 35, 36, 37, 38, 39, 40, 43, 44, 179, 185, 186, 188, 189, 190], [1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 32, 33, 34, 35, 36, 37, 38, 39, 43, 185, 186, 189, 190, 192], [4, 7, 8, 9, 10, 11, 12, 13, 32, 33, 34, 35, 37, 38, 185, 186, 189, 190, 192, 193, 194, 195, 196]]
neighbor_list_time=0.00s
place_water_time=0.26s
Number of triples: 6803
Number of placed waters: 4186
append_imput

{'filtered_gt_solvents': 3,
 'imputed_solvents': 106,
 'matching_solvents': 1,
 'recall': 0.3333333333333333}

In [97]:
example_result = recall_result("1ubq", recall_threshold=2.0, protein_clash_rad=2.5, solvent_clash_rad=0.0, max_hbond_length=3.5)
{
    "filtered_gt_solvents": example_result.num_waters_a,
    "imputed_solvents": example_result.num_waters_b,
    "matching_solvents": example_result.num_matching_waters,
    "recall": example_result.recall,
}

Number of candidate atoms: 223
Number of neighbor lists: 223
Neighbor lists: [[0, 1, 2, 3, 4, 7, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49], [0, 1, 2, 3, 4, 7, 8, 37, 38, 39, 40, 41, 42, 43, 44, 45, 47, 48, 185], [0, 1, 2, 3, 4, 7, 43, 44, 45, 46, 47, 49, 177, 178, 179, 182, 183, 185, 189, 191], [0, 1, 2, 3, 4, 5, 6, 7, 8, 37, 38, 39, 40, 41, 43, 44, 179, 182, 183, 185, 188], [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 37, 38, 39, 43, 44, 179, 182, 183, 185, 186, 187, 188, 189, 190, 191], [3, 4, 5, 6, 7, 8, 34, 35, 36, 37, 38, 41, 188], [3, 4, 5, 6, 7, 8, 36, 38, 185, 187, 188], [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 33, 34, 35, 36, 37, 38, 39, 40, 43, 44, 179, 185, 186, 188, 189, 190], [1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 32, 33, 34, 35, 36, 37, 38, 39, 43, 185, 186, 189, 190, 192], [4, 7, 8, 9, 10, 11, 12, 13, 32, 33, 34, 35, 37, 38, 185, 186, 189, 190, 192, 193, 194, 195, 196]]
neighbor_list_time=0.00s
place_water_time=0.26s
Number of triples: 6803
Number of placed waters: 4186
append_imput

{'filtered_gt_solvents': 3,
 'imputed_solvents': 106,
 'matching_solvents': 2,
 'recall': 0.6666666666666666}

In [82]:
import pandas as pd

ID_LIST = ["8Q41", "8BH8", "8BH9", "8HIG", "8Q40", "8K3D", "8SVA", "8SSU", "8PE3", "8GN3", "8GN4", "8K4L", "8Q43", "8H0L", "8TP8", "8SSQ"]

rows = []
for pdb_id in ID_LIST:
    result = recall_result(pdb_id, recall_threshold=1.0, protein_clash_rad=1.5, solvent_clash_rad=0.5, max_hbond_length=3.5)
    rows.append(
        {
            "pdb_id": pdb_id,
            "filtered_gt_solvents": result.num_waters_a,
            "imputed_solvents": result.num_waters_b,
            "matching_solvents": result.num_matching_waters,
            "recall": result.recall,
        }
    )

results_df = pd.DataFrame(rows).sort_values("recall", ascending=False)
results_df.style.format({"recall": "{:.3f}"}).hide(axis="index")


Number of candidate atoms: 2590
Number of neighbor lists: 2590
Neighbor lists: [[0, 1, 2, 3, 4, 5, 7, 69, 71, 72, 255, 258, 260, 262], [0, 1, 2, 3, 4, 5, 7, 8, 69, 71, 72, 73, 251, 255, 258, 259, 260, 261, 262, 263, 264, 266], [0, 1, 2, 3, 4, 5, 7, 69, 70, 71, 72, 73, 74, 75, 76, 177, 255], [0, 1, 2, 3, 4, 5, 69, 70, 71, 72, 73, 75, 177], [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 64, 66, 67, 69, 70, 71, 72, 73, 74, 260, 262, 263, 264], [0, 1, 2, 3, 4, 5, 7, 8, 9, 10, 64, 66, 67, 69, 70, 71, 72, 73, 74, 76, 262, 263, 264], [4, 6, 64, 66, 72, 264, 265], [0, 1, 2, 4, 5, 7, 8, 9, 10, 11, 64, 66, 67, 68, 69, 70, 71, 72, 73, 74, 76, 260, 262, 263, 264, 266, 268, 269], [1, 4, 5, 7, 8, 9, 10, 11, 12, 64, 66, 68, 69, 73, 74, 76, 262, 263, 264, 265, 266, 267, 268, 269, 270, 272], [4, 5, 7, 8, 9, 10, 11, 12, 13, 66, 67, 68, 69, 73, 74, 76, 77, 78, 79, 263, 266, 268, 269, 272]]
neighbor_list_time=0.01s
place_water_time=3.76s
Number of triples: 98027
Number of placed waters: 63067
append_imputed_solvents_time

pdb_id,filtered_gt_solvents,imputed_solvents,matching_solvents,recall
8Q41,44,14515,44,1.000
8BH8,1,7557,1,1.000
8GN4,6,1653,6,1.000
8GN3,9,1684,9,1.000
8TP8,1,11642,1,1.000
8Q40,49,14684,47,0.959
8Q43,55,14882,52,0.945
8K4L,34,6647,32,0.941
8PE3,62,12128,57,0.919
8K3D,20,4600,18,0.900


In [106]:
import pandas as pd

ID_LIST = ["8Q41", "8BH8", "8BH9", "8HIG", "8Q40", "8K3D", "8SVA", "8SSU", "8PE3", "8GN3", "8GN4", "8K4L", "8Q43", "8H0L", "8TP8", "8SSQ"]

rows = []
for pdb_id in ID_LIST:
    result = recall_result(pdb_id, recall_threshold=2.0, protein_clash_rad=2.2, solvent_clash_rad=3.0, max_hbond_length=3.5)
    rows.append(
        {
            "pdb_id": pdb_id,
            "filtered_gt_solvents": result.num_waters_a,
            "imputed_solvents": result.num_waters_b,
            "matching_solvents": result.num_matching_waters,
            "recall": result.recall,
            "residues": count_stripped_gt_residues(pdb_id),
        }
    )

results_df = pd.DataFrame(rows).sort_values("recall", ascending=False)
results_df.style.format({"recall": "{:.3f}"}).hide(axis="index")


Number of candidate atoms: 2590
Number of neighbor lists: 2590
Neighbor lists: [[0, 1, 2, 3, 4, 5, 7, 69, 71, 72, 255, 258, 260, 262], [0, 1, 2, 3, 4, 5, 7, 8, 69, 71, 72, 73, 251, 255, 258, 259, 260, 261, 262, 263, 264, 266], [0, 1, 2, 3, 4, 5, 7, 69, 70, 71, 72, 73, 74, 75, 76, 177, 255], [0, 1, 2, 3, 4, 5, 69, 70, 71, 72, 73, 75, 177], [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 64, 66, 67, 69, 70, 71, 72, 73, 74, 260, 262, 263, 264], [0, 1, 2, 3, 4, 5, 7, 8, 9, 10, 64, 66, 67, 69, 70, 71, 72, 73, 74, 76, 262, 263, 264], [4, 6, 64, 66, 72, 264, 265], [0, 1, 2, 4, 5, 7, 8, 9, 10, 11, 64, 66, 67, 68, 69, 70, 71, 72, 73, 74, 76, 260, 262, 263, 264, 266, 268, 269], [1, 4, 5, 7, 8, 9, 10, 11, 12, 64, 66, 68, 69, 73, 74, 76, 262, 263, 264, 265, 266, 267, 268, 269, 270, 272], [4, 5, 7, 8, 9, 10, 11, 12, 13, 66, 67, 68, 69, 73, 74, 76, 77, 78, 79, 263, 266, 268, 269, 272]]
neighbor_list_time=0.01s
place_water_time=3.72s
Number of triples: 98027
Number of placed waters: 63067
append_imputed_solvents_time

pdb_id,filtered_gt_solvents,imputed_solvents,matching_solvents,recall,residues
8BH8,1,494,1,1.000,558
8BH9,18,459,18,1.000,550
8GN4,6,83,6,1.000,91
8Q40,49,943,44,0.898,887
8GN3,9,93,8,0.889,91
8PE3,62,771,55,0.887,786
8K4L,34,429,29,0.853,492
8Q41,44,937,37,0.841,897
8Q43,55,924,45,0.818,897
8HIG,5,240,4,0.800,552


In [107]:
results_df["ratio_imputed_to_residues"] = results_df["imputed_solvents"] / results_df["residues"]

In [108]:
results_df

,pdb_id,filtered_gt_solvents,imputed_solvents,matching_solvents,recall,residues,ratio_imputed_to_residues
1,8BH8,1,494,1,1.000000,558,0.885305
2,8BH9,18,459,18,1.000000,550,0.834545
10,8GN4,6,83,6,1.000000,91,0.912088
4,8Q40,49,943,44,0.897959,887,1.063134
9,8GN3,9,93,8,0.888889,91,1.021978
8,8PE3,62,771,55,0.887097,786,0.980916
11,8K4L,34,429,29,0.852941,492,0.871951
0,8Q41,44,937,37,0.840909,897,1.044593
12,8Q43,55,924,45,0.818182,897,1.030100
3,8HIG,5,240,4,0.800000,552,0.434783
